REDES BAYESIANAS

¿Cuál es la probabilidad de aprobar el curso?

In [2]:
!pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 13.8 MB/s eta 0:00:00


In [3]:
# libreria de redes bayesianas

# crear la red bayesiana
from pgmpy.models import DiscreteBayesianNetwork

# crear una tabla de probabilidades condicionales
from pgmpy.factors.discrete import TabularCPD

# realizar inferencia en la red bayesiana
from pgmpy.inference import VariableElimination

/usr/local/lib/python3.13/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


In [4]:
# crear la red bayesiana

model = DiscreteBayesianNetwork([
    ("Aprobar", "horas_estudio"),
    ("Aprobar", "Asistencia"),
    ("Aprobar", "nota_examen"),
])

In [5]:
# crear la tabla de probabilidades condicionales para la variable Aprobar

# que puede 30% de probabilidad de que apruebe y 70% de probabilidad de que no apruebe

cpd_aprobar = TabularCPD(
    variable="Aprobar",
    variable_card=2,
    values=[[0.3], [0.7]],
    state_names={"Aprobar": ["Sí", "No"]}
)

In [6]:
# crear la tabla de probabilidades para la variable horas_estudio
# es un hipotesis de incertidumbre,
# que puede 80% de probabilidad de que el pronóstico sea "Sí" si apruebe
# y 20% de probabilidad de que el pronóstico sea "No" si apruebe
cpd_horas_estudio = TabularCPD(
    variable="horas_estudio",
    variable_card=2,
    values=[[0.8, 0.2], [0.2, 0.8]],
    evidence=["Aprobar"],
    evidence_card=[2],
    state_names={"horas_estudio": ["Sí", "No"], "Aprobar": ["Sí", "No"]}
)

In [7]:
# crear la tabla de probabilidades condicionales para asistencia
# que puede 90% de probabilidad de que se lleve asistencia
# si aprueba y 10% de probabilidad de asistencia si no inasistencia
cpd_asistencia = TabularCPD(
    variable="Asistencia",
    variable_card=2,
    values=[[0.9, 0.1], [0.1, 0.9]],
    evidence=["Aprobar"],
    evidence_card=[2],
    state_names={"Asistencia": ["Sí", "No"], "Aprobar": ["Sí", "No"]}
)

In [8]:
cpd_nota_examen = TabularCPD(
    variable="nota_examen",
    variable_card=2,
    values=[[0.9, 0.1], [0.1, 0.9]],
    evidence=["Aprobar"],
    evidence_card=[2],
    state_names={"nota_examen": ["Sí", "No"], "Aprobar": ["Sí", "No"]}
)

In [9]:
# agregar al modelo las tablas de probabilidades condicionales
model.add_cpds(cpd_aprobar, cpd_horas_estudio, cpd_asistencia, cpd_nota_examen)
model.check_model()

True

In [10]:
print("Modelo de red bayesiana creado y verificado correctamente.")
print(model.edges())

Modelo de red bayesiana creado y verificado correctamente.
[('Aprobar', 'horas_estudio'), ('Aprobar', 'Asistencia'), ('Aprobar', 'nota_examen')]


In [11]:
# Inferencia en la red bayesiana4
# motor o proceso de inferencia para al grafo
inferencia = VariableElimination(model)
resultado_aprobar = inferencia.query(
    variables=["Aprobar"],
    evidence={"horas_estudio": "Sí", "Asistencia": "Sí", "nota_examen": "Sí"}
)
print("Probabilidad de que apruebe  dado que las horas_estudio es 'Sí' y se lleve asistencia y nota_examen:")
print(resultado_aprobar)

Probabilidad de que apruebe  dado que las horas_estudio es 'Sí' y se lleve asistencia y nota_examen:
+-------------+----------------+
| Aprobar     |   phi(Aprobar) |
+=============+================+
| Aprobar(Sí) |         0.9928 |
+-------------+----------------+
| Aprobar(No) |         0.0072 |
+-------------+----------------+


In [12]:
resultado_aprobar_2 = inferencia.query(
    variables=["Aprobar"])

print("Probabilidad de que apruebe sin evidencia:")
print(resultado_aprobar_2)

Probabilidad de que apruebe sin evidencia:
+-------------+----------------+
| Aprobar     |   phi(Aprobar) |
+=============+================+
| Aprobar(Sí) |         0.3000 |
+-------------+----------------+
| Aprobar(No) |         0.7000 |
+-------------+----------------+
